# N103 · 最长递增子序列与耐心排序

**目标：** 从平方DP推导最小尾值压缩并恢复一条解。

**先修：** N042, N098, N037。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 严格/非严格；tails不等于答案序列；lower/upper bound；前驱恢复。

**配套讲解来源：** [同名逐章意见](../../comment/103_lis_and_ordered_sequences.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

**最长递增子序列（LIS）**：从数组里挑若干个元素（保持原有先后顺序，但不要求连续），使它们严格递增，问最多能挑多少个。比如输入 `[10,9,2,5,3,7,101,18]`，输出 `4`，因为 `2,3,7,18`（或 `2,5,7,101` 等）长度为 4，没有更长的了。注意"子序列"允许跳着选，这和"子数组必须连续"不同。

本章还有三个衍生问题：

1. **恢复一条具体的解**：光知道长度是 4 不够，还要把 `[2,3,7,18]` 这条序列本身拿出来，对应接口 `reconstruct_lis`。
2. **俄罗斯套娃信封**：信封 `(w,h)` 能装进另一个信封当且仅当宽和高都严格更小，问最多能套几层。比如 `[[5,4],[6,4],[6,7],[2,3]]` 输出 `3`，链条是 `(2,3)→(5,4)→(6,7)`。
3. **从平方算法到 O(n log n)**：朴素 DP 要 O(n²)，本章用"耐心排序 + 二分"把它压到 O(n log n)，这是本章的主要推导内容。

一个容易被忽视的口径问题：本章全程做**严格递增**（相等不算递增），所以 `[2,2,2]` 的 LIS 长度是 1，不是 3。

## 2. 基础知识：先读懂这些词

- **子序列**：保持相对顺序、允许跳过元素的挑选结果。`2,5,101` 是 `[10,9,2,5,3,7,101,18]` 的子序列，虽然它们在原数组里不相邻。
- **以 i 结尾的 LIS（平方 DP 状态）**：`dp[i]` = 必须选第 i 个元素时，以它结尾的最长递增子序列长度。转移是看前面所有比它小的元素 j，取 `dp[j]` 的最大值加一。这和 N099 的"以当前位置结尾"是同一招。
- **耐心排序（patience sorting）与 tails 数组**：换个角度存信息——`tails[k]` 表示"长度为 k+1 的递增子序列中，结尾元素的最小可能值"。为什么存最小尾值？因为结尾越小，后面的数越容易接得上，"未来延伸能力"只强不弱，所以只需要记住每个长度的最佳（最小）尾巴。
- **tails 不等于答案序列**：这是本章最大的坑。`tails` 在运行过程中不断被后来者替换，它只是"各长度的最优尾值档案"，最终内容 `[2,3,7,18]` 恰好递增只是因为档案本身有序，它**不一定**是原数组的某个真实子序列。想要真的解，必须额外记前驱。
- **lower_bound / upper_bound（`bisect_left` / `bisect_right`）**：`bisect_left` 找"第一个 ≥ x 的位置"，`bisect_right` 找"第一个 > x 的位置"。求严格递增用 left：遇到相等元素时定位到已有元素身上并替换它，长度不涨；求非递减（允许相等）才换成 right。两者不能随意互换。
- **前驱（parent）恢复**：把每个元素接上去时，记下"我接在哪个下标后面"，最后从最长链的末端沿 parent 指针一路走回开头，就还原出一条真实子序列。
- **支配关系**：长度相同（甚至更长）而尾值更小的档案，永远不比尾值大的差——这是 tails 数组始终有序、可以二分查找的根本原因。
- **二维偏序（信封问题）**：两个维度都要严格递增。技巧是第一维普通升序排好，第二维在第一维相同时**降序**排，然后对第二维求严格 LIS。降序的作用是让同宽的信封在高度序列里互相"顶掉"，不可能被同时选中。

## 3. 思路推导：从小例子开始

**Step 1（平方 DP，先建立直觉）**：`dp[i]` = 以第 i 个元素结尾的最长链长度。它只能接在"值比它小"的前辈后面，所以扫一遍所有 j<i 且 `nums[j]<nums[i]`，取最大的 `dp[j]` 加一；前面没有更小的就自立门户为 1。答案取所有 `dp[i]` 的最大值。手算 `[1,3,2]`：dp[0]=1；dp[1]=1+dp[0]=2（3 接在 1 后）；dp[2]=1+dp[0]=2（2 也只能接在 1 后）；答案是 2。这个做法清晰但每个 i 都要回头扫一遍，共 O(n²)。

**Step 2（换一种信息：每个长度的最小尾值）**：我们其实不需要对每个 i 都记全信息，只需要回答"想接出长度 k+1，最省的尾巴是多少"。维护 `tails`：读到新数 x 时，用二分在 tails 里找第一个 ≥ x 的位置 p。若 p 撞到末尾，说明 x 比所有尾值都大，它能把最长链再延长一格，append；否则用 x 替换 `tails[p]`——长度 p+1 的链有了更小的尾值，档案升级，长度不变。

**Step 3（手演 tails 的演化）**：“运行示例”部分 用 `[10,9,2,5,3,7,101,18]` 演示了全过程：

| 读入 | 替换长度下标 p | 最小末值表 tails |
|---|---|---|
| 10 | 0 | [10] |
| 9 | 0 | [9] |
| 2 | 0 | [2] |
| 5 | 1 | [2,5] |
| 3 | 1 | [2,3] |
| 7 | 2 | [2,3,7] |
| 101 | 3 | [2,3,7,101] |
| 18 | 3 | [2,3,7,18] |

逐格看：10 先立档；9 和 2 依次把"长度 1 的最小尾值"从 10 压到 9 再压到 2（替换不改变长度，只是档案更优）；5 比所有尾值大，开出长度 2 的新档；3 把长度 2 的尾巴从 5 换成更小的 3；7 开长度 3；101 开长度 4；18 把长度 4 的尾巴从 101 换成 18。最终 `len(tails)=4` 就是 LIS 长度。注意 `[2,3,7,18]` 在这个例子里恰好是真子序列，但那是巧合——替换会把"不是同一链条"的值混进同一张表。

**Step 4（恢复一条真链）**：另存两个数组——`indices[k]` 是"当前长度 k+1 档案对应的原数组下标"，`parent[i]` 是"i 接在了谁后面"（即接上时长度 p 档案的下标 `indices[p-1]`）。处理完后从 `indices[-1]`（最长链的末项）沿 parent 走回 -1，收集起来反转就是一条真实子序列。

**Step 5（信封问题归约到 LIS）**：把信封按 (宽升序, 同宽时高降序) 排列，对高度序列求严格 LIS。宽度排序保证了"排在后面的宽不小于前面"，而严格 LIS 只会从左往右接；同宽时高降序保证同宽的两个不可能同时入选（高度在序列里是递减的，严格递增链最多取其中一个）。

## 4. 核心代码：lis_quadratic

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def lis_quadratic(nums):
    dp = [1] * len(nums)
    for i, x in enumerate(nums):
        dp[i] = 1 + max((dp[j] for j in range(i) if nums[j] < x), default=0)
    return max(dp, default=0)
```

### 逐段读懂代码

**函数一：`lis_quadratic`（平方 DP 基线）**

```python
def lis_quadratic(nums):
    dp = [1] * len(nums)
    for i, x in enumerate(nums):
        dp[i] = 1 + max((dp[j] for j in range(i) if nums[j] < x), default=0)
    return max(dp, default=0)
```

- `dp=[1]*len(nums)`：每个元素自带长度 1（只选自己）。
- `max((...), default=0)`：生成器遍历所有值小于 x 的前辈取最大 dp；一个前辈都没有时 max 落空，`default=0` 兜底，结果仍是 1。`nums[j]<x` 的严格小于保证了严格递增语义。
- `return max(dp,default=0)`：空数组时 dp 为空，`default=0` 让空输入返回 0，这是边界契约。

**函数二：`lis_length`（二分版）**

```python
def lis_length(nums):
    tails = []
    for x in nums:
        p = bisect_left(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)
```

- `p=bisect_left(tails,x)`：在有序的 tails 里二分找第一个 ≥ x 的位置，O(log n)。
- `if p==len(tails)`：x 比全部尾值都大，能延长最长链，append 追加（tails 末尾扩张一格）。
- `else: tails[p]=x`：x 顶替掉"长度 p+1 原来的尾值"，让该长度档案更优。长度不变、表仍有序。
- 末尾返回 `len(tails)`：表长就是 LIS 长度。注意它不返回 tails 本身，因为 tails 未必是真子序列。

**函数三：`reconstruct_lis`（带前驱恢复）**

```python
def reconstruct_lis(nums):
    tails = []
    indices = []
    parent = [-1] * len(nums)
    for i, x in enumerate(nums):
        p = bisect_left(tails, x)
        if p:
            parent[i] = indices[p - 1]
        if p == len(tails):
            tails.append(x)
            indices.append(i)
        else:
            tails[p] = x
            indices[p] = i
    out = []
    i = indices[-1] if indices else -1
    while i != -1:
        out.append(nums[i])
        i = parent[i]
    return out[::-1]
```

- `parent=[-1]*len(nums)`：前驱初始化为 −1，表示"链头，前面没人"。
- `if p: parent[i]=indices[p-1]`：p>0 说明 x 是接在"长度 p 的档案"后面成为长度 p+1，前驱就是那个档案的下标。注意这行在更新 indices **之前**执行，读到的是更新前的正确旧值。
- `indices[p]=i` / `indices.append(i)`：与 tails 同步维护"每个长度档案当前对应的原下标"。
- `i=indices[-1] if indices else -1`：从最长链末项出发（空输入时直接得空结果）。
- `while i!=-1` 循环沿 parent 回溯收集元素，`out[::-1]` 反转成从左到右的顺序。
- 拿 `[10,9,2,5,3,7,101,18]` 走一遍回溯：末项下标 7（值 18）→ parent 指向下标 5（值 7）→ 下标 4（值 3）→ 下标 2（值 2）→ −1 停，反转得 `[2,3,7,18]`，正是 “运行示例”部分 打印的那条"真实恢复序列"。

**函数四：`max_envelopes`（俄罗斯套娃）**

```python
def max_envelopes(envelopes):
    return lis_length([h for w, h in sorted(envelopes, key=lambda x: (x[0], -x[1]))])
```

- `sorted(envelopes,key=lambda x:(x[0],-x[1]))`：宽度升序；宽度相等时高度取负再排，等效于高度降序。这个负号是全函数的点睛之笔。
- 列表推导取出高度序列，直接复用 `lis_length`。同宽信封在高度序列中呈降序，严格递增链至多选中其中一个，"宽和高都必须严格更大"的两个条件被一次性满足。

## 5. 分段实现：按顺序运行

**本章接口：** `lis_quadratic(nums)`、`lis_length(nums)`、`reconstruct_lis(nums)`、`max_envelopes(envelopes)`

### 导入本章使用的库

In [1]:
from bisect import bisect_left

### lis_quadratic

In [2]:
def lis_quadratic(nums):
    dp = [1] * len(nums)
    for i, x in enumerate(nums):
        dp[i] = 1 + max((dp[j] for j in range(i) if nums[j] < x), default=0)
    return max(dp, default=0)

### lis_length

In [3]:
def lis_length(nums):
    tails = []
    for x in nums:
        p = bisect_left(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)

### reconstruct_lis

In [4]:
def reconstruct_lis(nums):
    tails = []
    indices = []
    parent = [-1] * len(nums)
    for i, x in enumerate(nums):
        p = bisect_left(tails, x)
        if p:
            parent[i] = indices[p - 1]
        if p == len(tails):
            tails.append(x)
            indices.append(i)
        else:
            tails[p] = x
            indices[p] = i
    out = []
    i = indices[-1] if indices else -1
    while i != -1:
        out.append(nums[i])
        i = parent[i]
    return out[::-1]

### max_envelopes

In [5]:
def max_envelopes(envelopes):
    return lis_length([h for w, h in sorted(envelopes, key=lambda x: (x[0], -x[1]))])

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[10,9,2,5,3,7,101,18]; tails=[]; rows=[]
for x in a:
    p=bisect_left(tails,x)
    if p==len(tails): tails.append(x)
    else: tails[p]=x
    rows.append((x,p,tails[:]))
show_table(['读入','替换长度下标','最小末值表'],rows)
print('真实恢复序列',reconstruct_lis(a))

读入,替换长度下标,最小末值表
10,0,[10]
9,0,[9]
2,0,[2]
5,1,"[2, 5]"
3,1,"[2, 3]"
7,2,"[2, 3, 7]"
101,3,"[2, 3, 7, 101]"
18,3,"[2, 3, 7, 18]"


真实恢复序列 [2, 3, 7, 18]


## 7. 正确性、适用条件与复杂度

长度相同的递增序列，末值较小者支配较大者。lower_bound找到第一个不小于x的位置，保持严格递增含义；前驱在处理i前记录，保证索引和数值同时递增。相同宽度高度降序后，严格高度LIS不可能选择同宽的两项。

**代价：** 平方版O(n²)时间O(n)空间；二分版O(n log n)时间O(n)空间；恢复额外O(n)前驱。严格递增与非递减的区别是lower_bound/upper_bound，不能随意互换。

### 展开理解

**平方 DP 为什么对**：以 i 结尾的最长链，去掉末项后一定是某个"以更小元素 j 结尾"的链。反过来，任何值小于 x 的前辈 j 的链都能接上 x。所以 dp[i] = 1 + 前辈中最长的链，这个公式把所有可能都算尽了；全局答案无非是"以某个 i 结尾"，取全体最大即可。

**tails 二分为什么对**：核心是"尾值小的档案不吃亏"——一条长度 k、尾值 3 的链，未来能接上的数，长度 k、尾值 5 的链也都能接，反之不然。所以对每个长度只需保留最小尾值，tails 按长度递增时尾值也严格递增（否则短链的尾值不小于长链，说明长链其实可由短链延伸，矛盾），保持有序才能二分。新元素 x 要么比所有尾值大（唯一能延长表的情况），要么恰好能改进某个长度的尾值——`bisect_left` 找到的 p 正是"第一个尾值 ≥ x"的档位，替换它不会破坏有序性。严格性由 `bisect_left` 保证：相等元素定位到已有的同值档案上，做替换、不增长度，所以 `[2,2,2]` 长度停在 1。若改成 `bisect_right`，相等元素会 append 或替换到更右的档，长度会涨，那算的就是"非递减"版本了。

**前驱恢复为什么对**：记录 parent 发生在"把 x 接成长度 p+1"的那一刻，前驱取的是当时长度 p 档案的下标，所以 parent 链上"下标递增、数值也递增"同时成立（下标来自更早处理的元素，数值严格小于由 bisect 语义保证）。沿着这条链收出来的必然是原数组的真实子序列，且长度等于最终表长。

**信封为什么对**：排序后，若信封 A 排在 B 前面且高度能接上（hA<hB），则要么 wA<wB（可套），要么 wA==wB——但同宽时高度降序，hA<hB 不可能出现。所以高度序列里每条严格递增链都对应一组宽、高双严格的套娃链，反之亦然，两个问题的最长长度相等。

**复杂度**：平方版是 n 个元素每个回头扫 n 个前辈，O(n²)：n=1000 时约一百万步，还行；n=10⁵ 时约一百亿步，明显太慢。二分版每个元素一次 O(log n) 的二分加一次赋值，总 O(n log n)：n=10⁵ 时约 10⁵×17 ≈ 一百七十万次操作，一瞬间完成。空间上都是 O(n)（dp、tails、indices、parent 各占一个线性数组）。

## 8. 单元测试：每个用例在检查什么

```python
assert lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4
```

- 正常值测试：LeetCode 300 的经典样例，可与第三节的 tails 演化表逐行核对，长度 4 来自 `[2,3,7,18]`。

```python
assert lis_length([2, 2, 2]) == 1
```

- 严格性用例：三个相等的数在严格递增口径下只能选一个。它专治把 `bisect_left` 误换成 `bisect_right` 的错误——那会把答案算成 3。

```python
assert max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
```

- 正常值测试：最优链条 `(2,3)→(5,4)→(6,7)`，排序后高度序列为 `[3,4,7,4]`，严格 LIS 长度 3。

```python
assert max_envelopes([[1, 1], [1, 2], [1, 3]]) == 1
```

- 同宽陷阱用例：三个信封宽度全是 1，任何两个都不能互相套（宽必须严格更大），答案只能是 1。如果同宽时高度误用升序，高度序列 `[1,2,3]` 会被算出 3，这条断言当场抓住它——这正是降序排高度存在的意义。

```python
for n in range(7):
    for a in product(range(3), repeat=n):
        b = reconstruct_lis(a)
        assert len(b) == lis_length(a) == lis_quadratic(a)
        assert all((x < y for x, y in zip(b, b[1:])))
        it = iter(a)
        assert all((any((x == y for y in it)) for x in b))
```

- 穷举对拍：长度 0 到 6、元素取 0..2 的全部数组（含大量重复值，专测严格口径）。对每个输入做四重检查：(1) 恢复出的序列长度 == 二分版长度 == 平方版长度（三个实现互为证人）；(2) 恢复序列自身严格递增；(3) 恢复序列真的是原数组的子序列——用迭代器 `it` 只许向前扫描地匹配每个元素，保证顺序保持且元素真实存在。

In [7]:
assert lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4

assert lis_length([2, 2, 2]) == 1

assert max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3

assert max_envelopes([[1, 1], [1, 2], [1, 3]]) == 1

from itertools import product

for n in range(7):
    for a in product(range(3), repeat=n):
        b = reconstruct_lis(a)
        assert len(b) == lis_length(a) == lis_quadratic(a)
        assert all((x < y for x, y in zip(b, b[1:])))
        it = iter(a)
        assert all((any((x == y for y in it)) for x in b))

print('N103: 所有本章断言通过')

N103: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 证明tails按长度保持最小结尾。

**练习 2：** 处理信封同宽时排序方向。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（证明 tails 按长度保持最小尾值）**：提示——你要论证的是不变量："任意时刻，tails[k] 是当前已读前缀中所有长度为 k+1 的递增子序列结尾的最小值"。分两种情况讨论：新元素 x 替换 `tails[p]` 时，为什么"x 确实能形成长度 p+1 的链"（提示：p 是第一个 ≥x 的位置，说明 `tails[p-1]<x`，把 x 接在那条链后面即可）；以及为什么替换后其它档位不受影响。再想反例角度：如果 tails 允许无序，二分会发生什么。用 `[3,1,2]` 这样的输入手写每一步的 tails 和"真实各长度最小尾值"对照。
- **练习 2（信封同宽排序方向）**：提示——先做错误实验：把 `key=lambda x:(x[0],-x[1])` 改成 `(x[0],x[1])`，拿 `[[1,1],[1,2],[1,3]]` 跑一下看它错误地返回几。然后回答：同宽降序时，同宽信封在高度序列里是什么关系（递减），严格递增链为什么至多选一个；再推广一步——如果题目把"严格更大"放宽成"大于等于"，排序和二分要各改哪里（提示：同宽改升序、`bisect_left` 换 `bisect_right`）。边界别忘了两个信封完全相同的输入。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
from bisect import bisect_left

def lis_quadratic(nums):
    dp = [1] * len(nums)
    for i, x in enumerate(nums):
        dp[i] = 1 + max((dp[j] for j in range(i) if nums[j] < x), default=0)
    return max(dp, default=0)

def lis_length(nums):
    tails = []
    for x in nums:
        p = bisect_left(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)

def reconstruct_lis(nums):
    tails = []
    indices = []
    parent = [-1] * len(nums)
    for i, x in enumerate(nums):
        p = bisect_left(tails, x)
        if p:
            parent[i] = indices[p - 1]
        if p == len(tails):
            tails.append(x)
            indices.append(i)
        else:
            tails[p] = x
            indices[p] = i
    out = []
    i = indices[-1] if indices else -1
    while i != -1:
        out.append(nums[i])
        i = parent[i]
    return out[::-1]

def max_envelopes(envelopes):
    return lis_length([h for w, h in sorted(envelopes, key=lambda x: (x[0], -x[1]))])

# 示例与回归测试
assert lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4

assert lis_length([2, 2, 2]) == 1

assert max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3

assert max_envelopes([[1, 1], [1, 2], [1, 3]]) == 1

from itertools import product

for n in range(7):
    for a in product(range(3), repeat=n):
        b = reconstruct_lis(a)
        assert len(b) == lis_length(a) == lis_quadratic(a)
        assert all((x < y for x, y in zip(b, b[1:])))
        it = iter(a)
        assert all((any((x == y for y in it)) for x in b))

print('N103: 所有本章断言通过')

N103: 所有本章断言通过


## 11. 代表题与迁移

- **300. Longest Increasing Subsequence（最长递增子序列）**：LIS 本体，练平方 DP 到 tails 二分的推导，对应 `lis_quadratic` 与 `lis_length`。
- **354. Russian Doll Envelopes（俄罗斯套娃信封）**：二维偏序归约到一维 LIS 的经典迁移，练"同宽高度降序"这个技巧，对应 `max_envelopes`。
- **673. Number of Longest Increasing Subsequence（最长递增子序列的个数）**：在 LIS 骨架上从"求长度"升级为"数方案"，需要把本章的 tails 档案扩展成"长度+计数"双信息（“题目映射”部分 标注为 extension，本章接口未直接实现，留给你迁移）。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。